In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv('../data/features_data.csv')
X = df.drop(columns=['Time_taken(min)'])
y = df['Time_taken(min)']
print(y.isnull().sum())
print(df.shape)

0
(43862, 12)


In [3]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=10
)

In [4]:
numeric_cols = [
    'distance_km',
    'prep_time_min',
    'multiple_deliveries',
    'Delivery_person_Age',
    'Delivery_person_Ratings'
]

categorical_cols = [
    'Road_traffic_density',
    'Type_of_vehicle',
    'Weatherconditions',
    'City',
    'Festival'
]
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ]
)
X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)

print(X_train.isna().sum())

distance_traffic           0
distance_km                0
prep_time_min              0
Road_traffic_density       0
multiple_deliveries        0
Delivery_person_Age        0
Weatherconditions          0
Delivery_person_Ratings    0
City                       0
Festival                   0
Type_of_vehicle            0
dtype: int64


03 — Entraînement et comparaison des modèles

In [5]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor

models = {
    "Régression Linéaire": LinearRegression(),
    "Régression Ridge": Ridge(),
    "Arbre de Décision": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42)
}

trained_models = {}
for name, model in models.items():
    model.fit(X_train_encoded, y_train)
    trained_models[name] = model
    print(f"Modèle '{name}' entraîné avec succès !")

Modèle 'Régression Linéaire' entraîné avec succès !
Modèle 'Régression Ridge' entraîné avec succès !
Modèle 'Arbre de Décision' entraîné avec succès !
Modèle 'Random Forest' entraîné avec succès !


Parmi les quatre modèles testés, le Random Forest présente les meilleures performances sur l'ensemble de test. Il obtient le MAE le plus faible (3,64 minutes), le RMSE le plus faible (4,66 minutes) et le R² le plus élevé (0,756). Il est donc sélectionné pour l'étape d'optimisation des hyperparamètres.

In [6]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

results = []

for name, model in trained_models.items():

    y_pred = model.predict(X_test_encoded)

    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    results.append({
        "Modèle": name,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R²": r2
    })

results_df = pd.DataFrame(results)

results_df

,Modèle,MAE,MSE,RMSE,R²
0,Régression Linéaire,4.900561,38.453658,6.201101,0.567873
1,Régression Ridge,4.900537,38.452894,6.201040,0.567882
2,Arbre de Décision,4.680953,38.767126,6.226325,0.564351
3,Random Forest,3.635783,21.702476,4.658592,0.756116


In [12]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV

# 1. Instantiate the estimator
rf = RandomForestRegressor(random_state=42)

# 2. Define the hyperparameter search space
param_grid = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [None, 10, 20, 30, 50],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None]
}

# 3. Configure RandomizedSearchCV
random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_grid,
    n_iter=20,
    cv=5,
    scoring="neg_root_mean_squared_error",
    random_state=42,
    n_jobs=-1
)

# 4. Fit to training data
best_model=random_search.fit(
    X_train_encoded,
    y_train
)

# from sklearn.ensemble import RandomForestRegressor
# from sklearn.model_selection import GridSearchCV

# # 1. Créer le modèle
# rf = RandomForestRegressor(random_state=42)

# # 2. Définir les hyperparamètres à tester
# param_grid = {/home/kawkaw/DeliveryETA-/.venv/lib/python3.14/site-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.

#     'n_estimators': [100, 200, 300, 500],
#     'max_depth': [None, 10, 20, 30, 50],
#     'min_samples_split': [2, 5, 10],
#     'min_samples_leaf': [1, 2, 4],
#     'max_features': ['sqrt', 'log2', None]
# }

# # 3. Configurer GridSearchCV
# grid_search = GridSearchCV(
#     estimator=rf,
#     param_grid=param_grid,
#     cv=5,
#     scoring="neg_root_mean_squared_error",
#     n_jobs=-1
# )

# # 4. Entraîner sur les données d'entraînement
# best_model = grid_search.fit(
#     X_train_encoded,
#     y_train
# )

/home/kawkaw/DeliveryETA-/.venv/lib/python3.14/site-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


In [13]:
print("Meilleurs paramètres :")
print(random_search.best_params_)

Meilleurs paramètres :
{'n_estimators': 500, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': 'log2', 'max_depth': 30}


In [14]:

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
best_rf = RandomForestRegressor(
    n_estimators=500,
    min_samples_split=10,
    min_samples_leaf=2,
    max_features='log2',
    max_depth=30,
    random_state=42,
    n_jobs=-1
)

best_rf.fit(X_train_encoded, y_train)
y_pred = best_rf.predict(X_test_encoded)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("MAE :", mae)
print("MSE :", mse)
print("RMSE :", rmse)
print("R² :", r2)

MAE : 3.54463253285347
MSE : 20.273849611462897
RMSE : 4.502649177035992
R² : 0.7721706499535325


In [15]:
best_rf.fit(X_train_encoded, y_train)
y_train_pred = best_rf.predict(X_train_encoded)
y_train_pred = best_rf.predict(X_train_encoded)
mae_train = mean_absolute_error(y_train, y_train_pred)
rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
r2_train = r2_score(y_train, y_train_pred)
print("TRAIN")
print("MAE :", mae_train)
print("RMSE :", rmse_train)
print("R² :", r2_train)

print("\nTEST")
print("MAE :", mae)
print("RMSE :", rmse)
print("R² :", r2)

TRAIN
MAE : 2.8079375384152505
RMSE : 3.591303984199822
R² : 0.8527391100937136

TEST
MAE : 3.54463253285347
RMSE : 4.502649177035992
R² : 0.7721706499535325


In [11]:
n = X_test_encoded.shape[0]
p = X_test_encoded.shape[1]
r2_adj = 1 - (1 - r2) * (n - 1) / (n - p - 1)
print("R² ajusté :", r2_adj)

R² ajusté : 0.7715717157837909
